In [5]:
import numpy as np
import pandas as pd
import re
import os
import time
import warnings
import json
from scipy.special import expit
from scipy.sparse import csr_matrix, hstack
from scipy.sparse.linalg import spsolve

warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
STATS_DIR  = '/Users/markstolte/Downloads/march-madness/starter_pack/data/team_game_stats'
MIYA_DIR   = '/Users/markstolte/Downloads/march-madness/evan-miya'

# ============================================================
# CONFIG
# ============================================================
BEST_KNOWN_CONFIG = {
    # --- Elo ratings core ---
    'REGRESSION_TO_MEAN': 0.3033,
    'K_START':            0.9524,
    'K_DECAY':            0.0301,
    'HOME_COURT_ADJ':     3.300,
    'MAX_UPDATE':         4.442,
    'OFF_DEF_SPLIT':      0.510,
    'DEFAULT_TEMPO':     68.0,

    # --- Adjustment parameters (sweep-optimized) ---
    'MIN_GAMES_ADJ':      11,      # min games before computing any adjustment
    'FORM_DECAY':         0.906,   # exponential decay for recency weighting

    # Opponent-strength slope
    'OPP_STRENGTH_SHRINK': 0.80,   # shrinkage toward zero (was 0.85)

    # Pace adjustment — disabled, confirmed noise by sweep
    'PACE_SHRINK':         1.00,   # 1.0 = fully shrunk = disabled

    # Mean-reversion (inverted form) — strongest signal
    'REVERSION_WEIGHT':    0.48,   # (was 0.10) — sweep optimal

    # Consistency weighting — disabled as spread adj, useful as bet filter
    'CONSISTENCY_BLEND':   0.0,    # 0 = disabled

    # Global cap on total adjustment magnitude
    'MAX_TOTAL_ADJ':       8.0,
}

# ============================================================
# NAME MAPPING
# ============================================================
MIYA_NAME_MAP = {
    "Albany": "UAlbany",
    "American": "American University",
    "Appalachian State": "App State",
    "Arkansas-Little Rock": "Little Rock",
    "College of Charleston": "Charleston",
    "Connecticut": "UConn",
    "Detroit": "Detroit Mercy",
    "Fort Wayne": "Purdue Fort Wayne",
    "Hawaii": "Hawai'i",
    "IU Indy": "IU Indianapolis",
    "Illinois-Chicago": "UIC",
    "Long Island": "Long Island University",
    "Louisiana-Lafayette": "Louisiana",
    "Louisiana-Monroe": "UL Monroe",
    "Maryland-Eastern Shore": "Maryland Eastern Shore",
    "McNeese State": "McNeese",
    "Miami (Fla.)": "Miami",
    "Missouri-Kansas City": "Kansas City",
    "Nicholls State": "Nicholls",
    "Penn": "Pennsylvania",
    "Prairie View": "Prairie View A&M",
    "Queens": "Queens University",
    "Saint Bonaventure": "St. Bonaventure",
    "Saint Francis (PA)": "St. Francis (PA)",
    "Sam Houston State": "Sam Houston",
    "San Jose State": "San José State",
    "Seattle": "Seattle U",
    "Southeastern Louisiana": "SE Louisiana",
    "Southern Mississippi": "Southern Miss",
    "St. Thomas (MN)": "St. Thomas-Minnesota",
    "Tennessee-Martin": "UT Martin",
    "Texas-Rio Grande Valley": "UT Rio Grande Valley",
}

NAME_MAP = {
    "Connecticut": "UConn",
    "Hawaii": "Hawai'i",
    "Long Island": "Long Island University",
    "Miami (Fla.)": "Miami",
}

FULL_NAME_MAP = {**NAME_MAP, **MIYA_NAME_MAP}

# ============================================================
# DATA LOADING
# ============================================================
def load_all_data():
    all_seasons = []
    for f in sorted(os.listdir(STATS_DIR)):
        if f.endswith('.csv'):
            year = int(f[:4])
            if year >= 2022:
                df = pd.read_csv(os.path.join(STATS_DIR, f))
                all_seasons.append(df)
    game_stats = pd.concat(all_seasons, ignore_index=True)
    game_stats['team']     = game_stats['team'].map(lambda t: FULL_NAME_MAP.get(t, t))
    game_stats['opponent'] = game_stats['opponent'].map(lambda t: FULL_NAME_MAP.get(t, t))

    before = len(game_stats)
    game_stats = game_stats[
        game_stats['conference'].notna() & game_stats['opponentConference'].notna()
    ].reset_index(drop=True)
    print(f"  D1 filter: {before:,} → {len(game_stats):,} games "
          f"({before - len(game_stats):,} non-D1 removed)")

    miya_seasons = []
    for f in sorted(os.listdir(MIYA_DIR)):
        if not f.endswith('.csv'):
            continue
        match = re.search(r'(\d{2})-(\d{2})', f)
        if not match:
            continue
        season = 2000 + int(match.group(2))
        df = pd.read_csv(os.path.join(MIYA_DIR, f))
        df['season'] = season
        miya_seasons.append(df)
    miya_all = pd.concat(miya_seasons, ignore_index=True)
    miya_all['team_mapped'] = miya_all['team'].map(lambda t: FULL_NAME_MAP.get(t, t))

    return game_stats, miya_all


# ============================================================
# PRIORS
# ============================================================
def build_priors(miya_all, game_stats, config):
    RTM  = config['REGRESSION_TO_MEAN']
    DTMP = config.get('DEFAULT_TEMPO', 68.0)

    rank_to_bpr_curves = {}
    for season in sorted(miya_all['season'].unique()):
        sdf = miya_all[miya_all['season'] == season].copy()
        sdf = sdf.sort_values('bpr', ascending=False).reset_index(drop=True)
        sdf['rank_position'] = sdf.index + 1
        rank_to_bpr_curves[season] = dict(zip(sdf['rank_position'], sdf['bpr']))

    def rank_to_bpr(rr, ref):
        curve = rank_to_bpr_curves.get(ref, {})
        if not curve:
            return 0.0
        rank = max(1, min(int(round(rr)), max(curve.keys())))
        if rank in curve:
            return curve[rank]
        avail = sorted(curve.keys())
        for i, r in enumerate(avail):
            if r >= rank:
                if i == 0:
                    return curve[r]
                lo, hi = avail[i - 1], r
                frac = (rank - lo) / (hi - lo)
                return curve[lo] * (1 - frac) + curve[hi] * frac
        return curve[avail[-1]]

    priors = {}
    for gs_season in sorted(game_stats['season'].unique()):
        cur = miya_all[miya_all['season'] == gs_season]
        prv = miya_all[miya_all['season'] == gs_season - 1]
        ref = gs_season - 1
        has_rr = 'roster_rank' in cur.columns

        if has_rr and ref in rank_to_bpr_curves:
            for _, row in cur.iterrows():
                team = row['team_mapped']
                rr   = row.get('roster_rank', np.nan)
                if pd.notna(rr) and rr > 0:
                    raw = rank_to_bpr(rr, ref)
                    p   = raw * (1 - RTM)
                    priors[(team, gs_season)] = {
                        'obpr': p * 0.55, 'dbpr': p * 0.45,
                        'tempo': row.get('tempo', DTMP) if pd.notna(row.get('tempo')) else DTMP,
                        'source': 'roster_rank',
                    }

        if len(prv) > 0:
            mo = prv['obpr'].mean()
            md = prv['dbpr'].mean()
            for _, row in prv.iterrows():
                team = row['team_mapped']
                if (team, gs_season) in priors:
                    continue
                priors[(team, gs_season)] = {
                    'obpr': row['obpr'] * (1 - RTM) + mo * RTM,
                    'dbpr': row['dbpr'] * (1 - RTM) + md * RTM,
                    'tempo': row.get('tempo', DTMP) if pd.notna(row.get('tempo')) else DTMP,
                    'source': 'prior_bpr',
                }
    return priors


# ============================================================
# INCREMENTAL RATINGS ENGINE (unchanged from previous version)
# ============================================================
def _run_ratings_core(gs, priors, config, ratings, processed):
    """
    Core ratings loop. Mutates `ratings` and `processed` in place.
    Returns gs with pregame columns + residuals.
    """
    DTMP = config.get('DEFAULT_TEMPO', 68.0)
    ODS  = config.get('OFF_DEF_SPLIT', 0.5)

    gs = gs.sort_values(['season', 'startDate', 'gameId']).copy()
    if 'mov' not in gs.columns:
        gs['mov'] = gs['teamStats_points_total'] - gs['opponentStats_points_total']
    if 'pace' not in gs.columns:
        for col in ['teamStats_possessions', 'teamStats_fourFactors_possessions', 'pace']:
            if col in gs.columns:
                gs['pace'] = gs[col]; break
        else:
            gs['pace'] = DTMP

    def get_r(team, season):
        key = (team, season)
        if key not in ratings:
            pr = priors.get(key, {'obpr': 0.0, 'dbpr': 0.0, 'tempo': DTMP})
            ratings[key] = {
                'obpr': pr['obpr'], 'dbpr': pr['dbpr'],
                'tempo': pr.get('tempo', DTMP), 'games': 0
            }
        return ratings[key]

    def get_k(g):
        return config['K_START'] / (1 + g * config['K_DECAY'])

    n = len(gs)
    pg = {c: np.full(n, np.nan) for c in [
        'pregame_obpr', 'pregame_dbpr', 'pregame_bpr', 'pregame_tempo', 'pregame_games',
        'opp_pregame_obpr', 'opp_pregame_dbpr', 'opp_pregame_bpr',
        'opp_pregame_tempo', 'opp_pregame_games',
    ]}

    idx_map = {idx: pos for pos, idx in enumerate(gs.index)}
    MU = config['MAX_UPDATE']

    for idx, row in gs.iterrows():
        pos = idx_map[idx]
        s, t, o = row['season'], row['team'], row['opponent']
        mov = row['mov']
        if pd.isna(mov):
            continue
        rt, ro = get_r(t, s), get_r(o, s)

        pg['pregame_obpr'][pos]      = rt['obpr']
        pg['pregame_dbpr'][pos]      = rt['dbpr']
        pg['pregame_bpr'][pos]       = rt['obpr'] + rt['dbpr']
        pg['pregame_tempo'][pos]     = rt['tempo']
        pg['pregame_games'][pos]     = rt['games']
        pg['opp_pregame_obpr'][pos]  = ro['obpr']
        pg['opp_pregame_dbpr'][pos]  = ro['dbpr']
        pg['opp_pregame_bpr'][pos]   = ro['obpr'] + ro['dbpr']
        pg['opp_pregame_tempo'][pos] = ro['tempo']
        pg['opp_pregame_games'][pos] = ro['games']

        gk = (tuple(sorted([t, o])), s, row.get('startDate', ''))
        if gk in processed:
            continue
        processed.add(gk)

        ep = (rt['tempo'] + ro['tempo']) / 2 or DTMP
        em = ((rt['obpr'] + rt['dbpr']) - (ro['obpr'] + ro['dbpr'])) * (ep / 100)
        ih  = row.get('isHome', False)
        isn = row.get('neutralSite', False)
        hca = 0 if (isn or pd.isna(isn)) else (
            config['HOME_COURT_ADJ'] if ih else -config['HOME_COURT_ADJ']
        )
        em += hca

        surprise = mov - em
        if pd.isna(surprise):
            continue

        kt, ko = get_k(rt['games']), get_k(ro['games'])
        rt['obpr'] += np.clip(kt * surprise * ODS,       -MU, MU)
        rt['dbpr'] += np.clip(kt * surprise * (1 - ODS), -MU, MU)
        rt['games'] += 1
        ro['obpr'] -= np.clip(ko * surprise * ODS,       -MU, MU)
        ro['dbpr'] -= np.clip(ko * surprise * (1 - ODS), -MU, MU)
        ro['games'] += 1

        pace = row.get('pace', DTMP)
        if not np.isnan(pace) and pace > 0:
            rt['tempo'] = rt['tempo'] * 0.9 + pace * 0.1
            ro['tempo'] = ro['tempo'] * 0.9 + pace * 0.1

    for col, arr in pg.items():
        gs[col] = arr

    gs['expected_per100'] = gs['pregame_bpr'] - gs['opp_pregame_bpr']
    gs['expected_poss']   = (gs['pregame_tempo'] + gs['opp_pregame_tempo']) / 2

    gs['hca'] = 0.0
    if 'neutralSite' in gs.columns and 'isHome' in gs.columns:
        gs.loc[(gs['isHome'] == True) & (gs['neutralSite'] != True), 'hca'] =  config['HOME_COURT_ADJ']
        gs.loc[(gs['isHome'] == False) & (gs['neutralSite'] != True), 'hca'] = -config['HOME_COURT_ADJ']

    gs['expected_margin'] = gs['expected_per100'] * (gs['expected_poss'] / 100) + gs['hca']
    gs['residual']        = gs['mov'] - gs['expected_margin']

    return gs


def run_ratings_full(gs, priors, config):
    """Run from scratch. Returns (rated_df, ratings_dict, processed_set)."""
    ratings = {}
    processed = set()
    rated = _run_ratings_core(gs, priors, config, ratings, processed)
    return rated, ratings, processed


def run_ratings_incremental(new_gs, priors, config, ratings, processed):
    """Continue from existing state. Returns rated new games only."""
    return _run_ratings_core(new_gs, priors, config, ratings, processed)


# ============================================================
# NEW ADJUSTMENT SYSTEM
#
# Each adjustment is computed per-team from their game history,
# uses exactly ONE parameter per team (a single slope or mean),
# and is heavily shrunk toward zero.
#
# All adjustments are computed from the accumulated rated_df
# (prior games only) and stored in lookup dicts.
# ============================================================

def compute_team_adjustments(rated_df, config):
    """
    From a rated DataFrame of prior games, compute per-team adjustment
    signals. Returns a dict of dicts:
    
    {
        (team, season): {
            'opp_strength_slope': float,  # positive = plays UP to competition
            'pace_slope':         float,  # positive = better in fast games
            'form':               float,  # recent weighted residual mean
            'consistency':        float,  # 0-1 scale, 1 = very consistent
            'n_games':            int,
        }
    }
    """
    MIN_G    = config['MIN_GAMES_ADJ']
    FD       = config['FORM_DECAY']
    OPP_SHR  = config['OPP_STRENGTH_SHRINK']
    PACE_SHR = config['PACE_SHRINK']

    adjustments = {}

    for (team, season), grp in rated_df.groupby(['team', 'season']):
        grp = grp.sort_values('startDate')
        n = len(grp)
        if n < MIN_G:
            continue

        resids  = grp['residual'].values
        opp_bpr = grp['opp_pregame_bpr'].values
        pace    = grp['expected_poss'].values
        my_tempo = grp['pregame_tempo'].values

        valid = np.isfinite(resids) & np.isfinite(opp_bpr) & np.isfinite(pace)
        if valid.sum() < MIN_G:
            continue

        r  = resids[valid]
        ob = opp_bpr[valid]
        p  = pace[valid]
        mt = my_tempo[valid]
        nv = len(r)

        # Recency weights
        wts = np.array([FD ** (nv - 1 - j) for j in range(nv)])
        wts = wts / wts.sum()

        # ----- Opponent Strength Slope -----
        # Weighted regression of residual on opponent BPR (centered)
        ob_c = ob - np.average(ob, weights=wts)
        w_var_ob = np.average(ob_c ** 2, weights=wts)
        if w_var_ob > 1e-6:
            w_cov = np.average(r * ob_c, weights=wts)
            raw_slope = w_cov / w_var_ob
            # Shrink toward zero
            opp_slope = raw_slope * (1 - OPP_SHR)
        else:
            opp_slope = 0.0

        # ----- Pace Slope -----
        # Weighted regression of residual on pace deviation from team's own tempo
        pace_dev = p - mt
        pace_dev_c = pace_dev - np.average(pace_dev, weights=wts)
        w_var_pd = np.average(pace_dev_c ** 2, weights=wts)
        if w_var_pd > 1e-6:
            w_cov_p = np.average(r * pace_dev_c, weights=wts)
            raw_pace_slope = w_cov_p / w_var_pd
            pace_slope = raw_pace_slope * (1 - PACE_SHR)
        else:
            pace_slope = 0.0

        # ----- Form (weighted mean residual) -----
        form = float(np.average(r, weights=wts))

        # ----- Consistency (inverse of residual std, normalized) -----
        w_resid_var = np.average((r - form) ** 2, weights=wts)
        resid_std = np.sqrt(max(w_resid_var, 1e-6))
        # We'll normalize consistency later across all teams; for now store std
        consistency_std = resid_std

        adjustments[(team, season)] = {
            'opp_strength_slope': opp_slope,
            'pace_slope':         pace_slope,
            'form':               form,
            'consistency_std':    consistency_std,
            'n_games':            nv,
        }

    # Normalize consistency to 0-1 scale across all teams in each season
    # Lower std → higher consistency → closer to 1
    for season in set(s for (_, s) in adjustments.keys()):
        season_teams = {k: v for k, v in adjustments.items() if k[1] == season}
        if not season_teams:
            continue
        stds = [v['consistency_std'] for v in season_teams.values()]
        min_std = min(stds)
        max_std = max(stds)
        rng = max_std - min_std
        if rng < 1e-6:
            for k in season_teams:
                adjustments[k]['consistency'] = 0.5
        else:
            for k in season_teams:
                # Invert: low std → high consistency
                adjustments[k]['consistency'] = 1.0 - (
                    (adjustments[k]['consistency_std'] - min_std) / rng)

    return adjustments


def predict_game_adjustment(team_adj, opp_adj, opp_bpr, team_bpr,
                             expected_poss, team_tempo, opp_tempo, config):
    """
    Compute the total adjustment for a single game using the new system.
    
    Returns: (total_adj, opp_str_adj, pace_adj, reversion_adj, details_dict)
    """
    REV_W    = config['REVERSION_WEIGHT']
    CON_BL   = config['CONSISTENCY_BLEND']
    MAX_ADJ  = config['MAX_TOTAL_ADJ']

    # ----- 1. Opponent Strength -----
    # Team A's slope * opponent B's strength
    # If A has positive slope (plays up) and B is strong (high BPR),
    # A gets a positive adjustment
    opp_str_a = 0.0
    opp_str_b = 0.0
    if team_adj is not None:
        opp_str_a = team_adj['opp_strength_slope'] * opp_bpr
    if opp_adj is not None:
        opp_str_b = opp_adj['opp_strength_slope'] * team_bpr
    opp_str_adj = opp_str_a - opp_str_b

    # ----- 2. Pace -----
    # How much does pace deviate from each team's preference?
    pace_adj_a = 0.0
    pace_adj_b = 0.0
    if team_adj is not None:
        pace_dev_a = expected_poss - team_tempo
        pace_adj_a = team_adj['pace_slope'] * pace_dev_a
    if opp_adj is not None:
        pace_dev_b = expected_poss - opp_tempo
        pace_adj_b = opp_adj['pace_slope'] * pace_dev_b
    pace_adj = pace_adj_a - pace_adj_b

    # ----- 3. Mean Reversion (inverted form) -----
    # Negative: if a team has been running hot (positive form),
    # they're likely to regress → adjust prediction DOWN
    form_a = team_adj['form'] if team_adj is not None else 0.0
    form_b = opp_adj['form']  if opp_adj  is not None else 0.0
    reversion_adj = -REV_W * (form_a - form_b)

    # ----- 4. Consistency weighting -----
    # Scale total adjustment by average consistency of both teams
    # Inconsistent matchups → shrink adjustments further
    if CON_BL > 0:
        con_a = team_adj['consistency'] if team_adj is not None else 0.5
        con_b = opp_adj['consistency']  if opp_adj  is not None else 0.5
        avg_consistency = (con_a + con_b) / 2
        # Blend: total = total * (1 - CON_BL + CON_BL * avg_consistency)
        # When avg_consistency=1 (very consistent): multiplier ≈ 1
        # When avg_consistency=0 (very volatile): multiplier ≈ 1 - CON_BL
        consistency_mult = 1.0 - CON_BL + CON_BL * avg_consistency
    else:
        consistency_mult = 1.0

    raw_total = (opp_str_adj + pace_adj + reversion_adj) * consistency_mult
    total_adj = np.clip(raw_total, -MAX_ADJ, MAX_ADJ)

    details = {
        'opp_str_adj':   opp_str_adj,
        'pace_adj':      pace_adj,
        'reversion_adj': reversion_adj,
        'consistency_mult': consistency_mult,
        'raw_total':     raw_total,
    }

    return total_adj, details


# ============================================================
# EXPANDING-WINDOW PREDICTION ENGINE (incremental)
# ============================================================
def expanding_window_predict(all_games, test_game_ids, miya_all, config,
                              min_pregame_games=5):
    """
    Expanding-window predictions with incremental ratings and new adjustments.
    """
    DTMP = config.get('DEFAULT_TEMPO', 68.0)

    all_games = all_games.sort_values(['season', 'startDate', 'gameId']).copy()
    all_games['_date'] = pd.to_datetime(all_games['startDate'], format='ISO8601')
    if 'mov' not in all_games.columns:
        all_games['mov'] = (all_games['teamStats_points_total']
                            - all_games['opponentStats_points_total'])

    priors = build_priors(miya_all, all_games, config)

    min_date = all_games['_date'].min() - pd.Timedelta(days=7)
    max_date = all_games['_date'].max()
    week_starts = pd.date_range(min_date, max_date + pd.Timedelta(days=7), freq='7D')

    # Incremental state
    ratings   = {}
    processed = set()
    rated_parts = []

    predictions = []
    games_rated_so_far = 0

    # Cached adjustment lookups (rebuilt each week)
    current_adjustments = {}

    for i_week in range(len(week_starts)):
        ws = week_starts[i_week]
        ws_next = (week_starts[i_week + 1] if i_week + 1 < len(week_starts)
                   else max_date + pd.Timedelta(days=8))

        # --- 1) Feed new games into ratings ---
        new_games = all_games[
            (all_games['_date'] >= (week_starts[i_week - 1] if i_week > 0 else min_date)) &
            (all_games['_date'] < ws)
        ]
        if len(new_games) > 0:
            if games_rated_so_far == 0:
                rated_new, ratings, processed = run_ratings_full(
                    new_games, priors, config)
            else:
                rated_new = run_ratings_incremental(
                    new_games, priors, config, ratings, processed)

            rated_new['_date'] = pd.to_datetime(rated_new['startDate'], format='ISO8601')
            rated_parts.append(rated_new)
            games_rated_so_far += len(new_games)

        if games_rated_so_far < 30:
            continue

        # --- 2) Compute adjustments from all rated games so far ---
        all_rated = pd.concat(rated_parts, ignore_index=True)
        current_adjustments = compute_team_adjustments(all_rated, config)

        # --- 3) Predict test games in this week ---
        week_games = all_games[
            (all_games['_date'] >= ws) & (all_games['_date'] < ws_next)
        ]
        if test_game_ids is not None:
            week_games = week_games[week_games.index.isin(test_game_ids)]
        if len(week_games) == 0:
            continue

        for _, game in week_games.iterrows():
            team   = game['team']
            opp    = game['opponent']
            season = game['season']
            date   = game['startDate']

            rt = ratings.get((team, season))
            ro = ratings.get((opp, season))

            if rt is None:
                pr = priors.get((team, season),
                                {'obpr': 0.0, 'dbpr': 0.0, 'tempo': DTMP})
                rt = {'obpr': pr['obpr'], 'dbpr': pr['dbpr'],
                      'tempo': pr.get('tempo', DTMP), 'games': 0}
            if ro is None:
                pr = priors.get((opp, season),
                                {'obpr': 0.0, 'dbpr': 0.0, 'tempo': DTMP})
                ro = {'obpr': pr['obpr'], 'dbpr': pr['dbpr'],
                      'tempo': pr.get('tempo', DTMP), 'games': 0}

            if rt['games'] < min_pregame_games or ro['games'] < min_pregame_games:
                continue

            team_bpr = rt['obpr'] + rt['dbpr']
            opp_bpr  = ro['obpr'] + ro['dbpr']
            expected_per100 = team_bpr - opp_bpr
            expected_poss   = (rt['tempo'] + ro['tempo']) / 2

            ih  = game.get('isHome', False)
            isn = game.get('neutralSite', False)
            if isn or pd.isna(isn):
                hca = 0.0
            elif ih:
                hca = config['HOME_COURT_ADJ']
            else:
                hca = -config['HOME_COURT_ADJ']

            base_spread = expected_per100 * (expected_poss / 100) + hca

            # Look up team adjustments (computed from games BEFORE this week)
            # We need form computed from games strictly before this game's date,
            # but opp_strength and pace slopes use all prior data (they're stable)
            team_adj = current_adjustments.get((team, season))
            opp_adj  = current_adjustments.get((opp, season))

            total_adj, details = predict_game_adjustment(
                team_adj, opp_adj,
                opp_bpr, team_bpr,
                expected_poss, rt['tempo'], ro['tempo'],
                config
            )

            full_pred = base_spread + total_adj

            predictions.append({
                'gameId':      game['gameId'],
                'season':      season,
                'startDate':   game['startDate'],
                'team':        team,
                'opponent':    opp,
                'isHome':      ih,
                'neutralSite': isn,
                'mov':         game['mov'],
                'base_spread': base_spread,
                'opp_str_adj': details['opp_str_adj'],
                'pace_adj':    details['pace_adj'],
                'reversion_adj': details['reversion_adj'],
                'consistency_mult': details['consistency_mult'],
                'total_adj':   total_adj,
                'full_pred':   full_pred,
            })

    return predictions


# ============================================================
# MAIN EVALUATION
# ============================================================
LAMBDA_CORR    = 2.0
SIGMA_ESTIMATE = 11.0


def evaluate_config(config, game_stats, miya_all, eval_mode="walk_forward"):
    """Expanding-window evaluation with NO leakage."""
    priors_full = build_priors(miya_all, game_stats, config)
    gs_full, _, _ = run_ratings_full(game_stats.copy(), priors_full, config)
    gs_full = gs_full.sort_values(['season', 'startDate', 'gameId']).copy()

    if eval_mode == "walk_forward":
        test_mask = pd.Series(False, index=gs_full.index)
        for season in gs_full['season'].unique():
            sg = gs_full[gs_full['season'] == season].sort_values('startDate')
            cutoff = int(len(sg) * 0.80)
            test_mask.loc[sg.iloc[cutoff:].index] = True

        test_indices = set(gs_full[
            test_mask &
            (gs_full['pregame_games'] >= 5) &
            (gs_full['opp_pregame_games'] >= 5)
        ].index)
    else:
        test_indices = set(gs_full[
            (gs_full['pregame_games'] >= 5) &
            (gs_full['opp_pregame_games'] >= 5)
        ].index)

    if len(test_indices) == 0:
        return 999.0, 0.0, pd.DataFrame(), gs_full

    print(f"  Expanding-window eval: {len(test_indices):,} test games across "
          f"{gs_full.loc[list(test_indices), 'season'].nunique()} seasons")

    _t0 = time.time()
    predictions = expanding_window_predict(
        game_stats, test_indices, miya_all, config, min_pregame_games=5)
    print(f"  Predictions complete: {len(predictions):,} games in {time.time()-_t0:.1f}s")

    bt = pd.DataFrame(predictions)
    if len(bt) == 0:
        return 999.0, 0.0, pd.DataFrame(), gs_full

    bt['base_err'] = (bt['mov'] - bt['base_spread']).abs()
    bt['full_err'] = (bt['mov'] - bt['full_pred']).abs()

    mae  = bt['full_err'].mean()
    corr = bt['full_pred'].corr(bt['mov'])

    return mae, corr, bt, gs_full


# ============================================================
# DIAGNOSTICS
# ============================================================
def win_prob_from_spread(spread, sigma=SIGMA_ESTIMATE):
    return expit(-spread / sigma)

def brier_score(probs, outcomes):
    return np.mean((probs - outcomes) ** 2)

def run_diagnostics(bt):
    print("\n" + "=" * 80)
    print("PER-SEASON DIAGNOSTICS (expanding-window, no leakage)")
    print("=" * 80)
    print(f"{'Season':<8} {'N':>6} {'MAE':>7} {'RMSE':>7} {'Corr':>6} "
          f"{'SU%':>6} {'Brier':>7} {'BaseMAE':>8}")
    print("-" * 80)
    for season in sorted(bt['season'].unique()):
        s = bt[bt['season'] == season]
        n    = len(s)
        mae  = s['full_err'].mean()
        rmse = np.sqrt((s['full_err'] ** 2).mean())
        corr = s['full_pred'].corr(s['mov'])
        bm   = s['base_err'].mean()
        su   = ((s['full_pred'] > 0) == (s['mov'] > 0)).mean() * 100
        wp   = win_prob_from_spread(-s['full_pred'].values)
        out  = (s['mov'] > 0).astype(float).values
        bs   = brier_score(wp, out)
        print(f"{season:<8} {n:>6} {mae:>7.2f} {rmse:>7.2f} {corr:>6.3f} "
              f"{su:>5.1f}% {bs:>7.4f} {bm:>8.2f}")

    mae  = bt['full_err'].mean()
    rmse = np.sqrt((bt['full_err'] ** 2).mean())
    corr = bt['full_pred'].corr(bt['mov'])
    bm   = bt['base_err'].mean()
    su   = ((bt['full_pred'] > 0) == (bt['mov'] > 0)).mean() * 100
    wp   = win_prob_from_spread(-bt['full_pred'].values)
    out  = (bt['mov'] > 0).astype(float).values
    bs   = brier_score(wp, out)
    print("-" * 80)
    print(f"{'ALL':<8} {len(bt):>6} {mae:>7.2f} {rmse:>7.2f} {corr:>6.3f} "
          f"{su:>5.1f}% {bs:>7.4f} {bm:>8.2f}")


def run_adjustment_diagnostics(bt):
    """Detailed diagnostics for the new adjustment system."""
    print("\n" + "=" * 70)
    print("NEW ADJUSTMENT DIAGNOSTICS")
    print("=" * 70)

    # 1. Magnitude distributions
    print("\n--- Magnitude Distributions ---")
    for col in ['opp_str_adj', 'pace_adj', 'reversion_adj', 'total_adj']:
        vals = bt[col].dropna()
        print(f"  {col:<18s}: mean={vals.mean():+.3f}  std={vals.std():.3f}  "
              f"[{vals.quantile(0.05):+.2f}, {vals.quantile(0.95):+.2f}]")

    # 2. Correlation with MOV
    print("\n--- Correlation with MOV ---")
    for col in ['base_spread', 'opp_str_adj', 'pace_adj', 'reversion_adj',
                'total_adj', 'full_pred']:
        r = bt[col].corr(bt['mov'])
        print(f"  corr({col}, mov) = {r:.4f}")

    # 3. Correlation with base residual
    bt['base_resid'] = bt['mov'] - bt['base_spread']
    print("\n--- Correlation with Base Residual (does it fix errors?) ---")
    for col in ['opp_str_adj', 'pace_adj', 'reversion_adj', 'total_adj']:
        r = bt[col].corr(bt['base_resid'])
        print(f"  corr({col}, base_resid) = {r:.4f}")

    # 4. Direction accuracy
    print("\n--- Direction Accuracy (want > 50%) ---")
    for col in ['opp_str_adj', 'pace_adj', 'reversion_adj', 'total_adj']:
        nonzero = bt[bt[col].abs() > 0.01]
        if len(nonzero) == 0:
            continue
        correct = (np.sign(nonzero[col]) == np.sign(nonzero['base_resid'])).mean()
        print(f"  {col:<18s}: {correct:.1%} correct (n={len(nonzero):,})")

    # 5. Ablation
    print("\n--- Ablation ---")
    combos = [
        ('Base only',                  bt['base_spread']),
        ('Base + OppStr',              bt['base_spread'] + bt['opp_str_adj']),
        ('Base + Pace',                bt['base_spread'] + bt['pace_adj']),
        ('Base + Reversion',           bt['base_spread'] + bt['reversion_adj']),
        ('Base + OppStr + Pace',       bt['base_spread'] + bt['opp_str_adj'] + bt['pace_adj']),
        ('Base + All (full)',          bt['full_pred']),
    ]
    for label, pred in combos:
        err = (bt['mov'] - pred).abs().mean()
        corr = pred.corr(bt['mov'])
        su = ((pred > 0) == (bt['mov'] > 0)).mean() * 100
        print(f"  {label:<30s}  MAE={err:.3f}  Corr={corr:.4f}  SU={su:.1f}%")

    # 6. Optimal shrinkage on total_adj
    print("\n--- Optimal Weight on total_adj ---")
    best_w, best_mae = 0, (bt['mov'] - bt['base_spread']).abs().mean()
    for w in np.arange(-0.5, 3.05, 0.1):
        pred = bt['base_spread'] + w * bt['total_adj']
        mae = (bt['mov'] - pred).abs().mean()
        if mae < best_mae:
            best_mae = mae
            best_w = w
    print(f"  Best weight: {best_w:.1f}  (MAE={best_mae:.3f})")
    print(f"  Weight=0 (base only): MAE={(bt['mov']-bt['base_spread']).abs().mean():.3f}")
    print(f"  Weight=1 (current):   MAE={bt['full_err'].mean():.3f}")


# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    print("Loading data...")
    start_load = time.time()
    game_stats, miya_all = load_all_data()
    print(f"  Loaded in {time.time()-start_load:.1f}s — "
          f"{len(game_stats):,} games, {len(miya_all):,} Miya rows.\n")

    start_time = time.time()
    config = BEST_KNOWN_CONFIG.copy()

    print("Running model with new adjustment system...")
    mae, corr, bt, gs = evaluate_config(config, game_stats, miya_all,
                                         eval_mode="walk_forward")

    print("\n" + "=" * 70)
    print("MODEL EVALUATION COMPLETE")
    print("=" * 70)
    print(f"MAE:   {mae:.4f}")
    print(f"Corr:  {corr:.4f}")
    print(f"Score: {mae - LAMBDA_CORR * corr:.4f}")
    print(f"Total time: {time.time() - start_time:.0f}s")

    run_diagnostics(bt)
    run_adjustment_diagnostics(bt)

    out_dir = os.path.dirname(STATS_DIR)
    bt.to_csv(os.path.join(out_dir, 'model_predictions_v2.csv'), index=False)
    with open(os.path.join(out_dir, 'best_config_v2.json'), 'w') as f:
        json.dump(config, f, indent=2)
    print(f"\n✅ Saved predictions + config to {out_dir}")


    # ============================================================
    # VEGAS COMPARISON
    # ============================================================
    LINES_PATH = '/Users/markstolte/Downloads/march-madness/starter_pack/data/lines/lines_flat.csv'

    print("\nLoading clean flat lines...")
    lines = pd.read_csv(LINES_PATH)
    for col in ['homeTeam', 'awayTeam']:
        lines[col] = lines[col].map(lambda t: FULL_NAME_MAP.get(t, t))
    lines['_prov_rank'] = np.where(lines['provider'] == 'consensus', 0, 1)
    lines = lines.sort_values(['_prov_rank', 'provider']).drop_duplicates(
        subset=['gameId'], keep='first'
    ).drop(columns=['_prov_rank'])
    lines = lines.dropna(subset=['spread'])
    print(f"  {len(lines):,} unique games with spreads")

    print("\nBuilding expanding-window predictions for Vegas comparison...")
    _t1 = time.time()

    gs_for_vegas = game_stats.copy()
    gs_for_vegas['_date'] = pd.to_datetime(gs_for_vegas['startDate'], format='ISO8601')
    if 'mov' not in gs_for_vegas.columns:
        gs_for_vegas['mov'] = (gs_for_vegas['teamStats_points_total']
                                - gs_for_vegas['opponentStats_points_total'])
    games_with_lines = set(lines['gameId'].unique())
    vegas_test_indices = set(gs_for_vegas[gs_for_vegas['gameId'].isin(games_with_lines)].index)

    predictions = expanding_window_predict(
        gs_for_vegas, vegas_test_indices, miya_all, config, min_pregame_games=5)

    wf = pd.DataFrame(predictions)
    print(f"  Expanding-window predictions: {len(wf):,} games in {time.time()-_t1:.1f}s")

    # Merge with lines
    wf_merge = wf.copy()
    wf_merge['_home'] = np.where(
        wf_merge['isHome'] == True, wf_merge['team'], wf_merge['opponent'])
    wf_merge['_away'] = np.where(
        wf_merge['isHome'] == True, wf_merge['opponent'], wf_merge['team'])
    wf_merge['_date'] = pd.to_datetime(wf_merge['startDate'], format='ISO8601').dt.strftime('%Y-%m-%d')
    wf_merge['_merge_key'] = (wf_merge['_home'] + '|' + wf_merge['_away']
                               + '|' + wf_merge['_date'])

    lines['_date'] = pd.to_datetime(lines['startDate'], format='ISO8601').dt.strftime('%Y-%m-%d')
    lines['_merge_key'] = (lines['homeTeam'] + '|' + lines['awayTeam']
                            + '|' + lines['_date'])

    lines_s = lines[['_merge_key', 'spread', 'homeTeam', 'awayTeam']].rename(
        columns={'spread': 'vegas_spread'})

    merged = wf_merge.merge(lines_s, on='_merge_key', how='inner',
                             suffixes=('', '_lines'))
    merged = merged.drop_duplicates(subset='_merge_key', keep='first').copy()
    print(f"  Matched with lines: {len(merged):,} unique games")

    merged['is_home_row'] = (merged['team'] == merged['_home'])
    merged['vegas_margin'] = np.where(
        merged['is_home_row'], -merged['vegas_spread'], merged['vegas_spread'])

    vegas_su = ((merged['vegas_margin'] > 0) == (merged['mov'] > 0)).mean()
    print(f"  Vegas SU sanity check: {vegas_su:.1%} (expect ~70%)")

    merged['vegas_err'] = (merged['mov'] - merged['vegas_margin']).abs()
    merged['model_err'] = (merged['mov'] - merged['full_pred']).abs()
    merged['actual_winner']     = (merged['mov'] > 0)
    merged['model_pick_correct'] = (merged['full_pred'] > 0) == merged['actual_winner']
    merged['vegas_pick_correct'] = (merged['vegas_margin'] > 0) == merged['actual_winner']
    merged['actual_vs_spread']  = merged['mov'] - merged['vegas_margin']
    merged['model_vs_spread']   = merged['full_pred'] - merged['vegas_margin']
    merged['model_ats_correct'] = (
        np.sign(merged['model_vs_spread']) == np.sign(merged['actual_vs_spread'])
    ).astype(float)

    push_mask = (merged['actual_vs_spread'] == 0) | (merged['model_vs_spread'] == 0)
    ats_df = merged[~push_mask].copy()
    ats_df['edge'] = (ats_df['full_pred'] - ats_df['vegas_margin']).abs()

    print("\n" + "=" * 70)
    print("VEGAS vs MODEL — EXPANDING WINDOW (no leakage)")
    print("=" * 70)
    print(f"\n{'Season':<8} {'N':>6} {'VegasMAE':>9} {'ModelMAE':>9} "
          f"{'VegasSU%':>9} {'ModelSU%':>9}")
    print("-" * 60)
    for season in sorted(merged['season'].unique()):
        s  = merged[merged['season'] == season]
        n  = len(s)
        vm = s['vegas_err'].mean()
        mm = s['model_err'].mean()
        vsu = s['vegas_pick_correct'].mean() * 100
        msu = s['model_pick_correct'].mean() * 100
        print(f"{season:<8} {n:>6} {vm:>9.2f} {mm:>9.2f} "
              f"{vsu:>8.1f}% {msu:>8.1f}%")

    n  = len(merged)
    vm = merged['vegas_err'].mean()
    mm = merged['model_err'].mean()
    vsu = merged['vegas_pick_correct'].mean() * 100
    msu = merged['model_pick_correct'].mean() * 100
    print("-" * 60)
    print(f"{'ALL':<8} {n:>6} {vm:>9.2f} {mm:>9.2f} "
          f"{vsu:>8.1f}% {msu:>8.1f}%")

    print("\n" + "=" * 70)
    print("MODEL ATS BY MINIMUM EDGE (expanding window, no leakage)")
    print("  Edge = |model_pred - vegas_margin|")
    print("  Break-even with -110 vig: 52.4%")
    print("=" * 70)
    print(f"\n{'Min Edge':>10} {'ATS%':>7} {'Record':>12} {'N':>6}")
    print("-" * 50)
    for min_edge in [0, 1, 2, 3, 5, 7, 10, 15]:
        sub = ats_df[ats_df['edge'] >= min_edge]
        if len(sub) == 0:
            continue
        wins   = int(sub['model_ats_correct'].sum())
        losses = len(sub) - wins
        pct    = sub['model_ats_correct'].mean() * 100
        print(f"  >= {min_edge:<4}  {pct:>6.1f}%  {wins:>5}-{losses:<5}  {len(sub):>5}")

    print(f"\n{'Edge Range':>12} {'ATS%':>7} {'Record':>12} {'N':>6}")
    print("-" * 50)
    for lo, hi in [(0,1),(1,2),(2,3),(3,5),(5,7),(7,10),(10,15),(15,999)]:
        sub = ats_df[(ats_df['edge'] >= lo) & (ats_df['edge'] < hi)]
        if len(sub) == 0:
            continue
        wins   = int(sub['model_ats_correct'].sum())
        losses = len(sub) - wins
        pct    = sub['model_ats_correct'].mean() * 100
        label  = f"{lo}-{hi}" if hi < 999 else f"{lo}+"
        print(f"  {label:>8}  {pct:>6.1f}%  {wins:>5}-{losses:<5}  {len(sub):>5}")

    print()
    print(f"  Vegas MAE: {vm:.2f}  |  Model MAE: {mm:.2f}")
    print(f"  Vegas SU:  {vsu:.1f}%  |  Model SU:  {msu:.1f}%")
    print(f"  Break-even ATS (with -110 vig): 52.4%")

Loading data...
  D1 filter: 61,375 → 56,204 games (5,171 non-D1 removed)
  Loaded in 1.5s — 56,204 games, 1,810 Miya rows.

Running model with new adjustment system...
  Expanding-window eval: 11,242 test games across 5 seasons
  Predictions complete: 11,242 games in 254.6s

MODEL EVALUATION COMPLETE
MAE:   9.7461
Corr:  0.4905
Score: 8.7651
Total time: 262s

PER-SEASON DIAGNOSTICS (expanding-window, no leakage)
Season        N     MAE    RMSE   Corr    SU%   Brier  BaseMAE
--------------------------------------------------------------------------------
2022       2251    9.60   12.12  0.476  68.5%  0.2008     9.82
2023       2297    9.82   12.35  0.485  66.2%  0.2092    10.06
2024       2302   10.02   12.43  0.486  65.9%  0.2094    10.31
2025       2303    9.45   12.09  0.540  68.3%  0.1972     9.63
2026       2089    9.85   12.50  0.460  63.8%  0.2180     9.99
--------------------------------------------------------------------------------
ALL       11242    9.75   12.30  0.491  66.

In [6]:
"""
Pace & Consistency as BET FILTERS (not spread adjustments).

Hypothesis: These signals don't help predict the spread, but they DO
help identify WHEN our predictions are more reliable.

- Consistent teams → our model's edge is more stable → better ATS
- Pace-matched games → less chaos → spread predictions more accurate
- Combined: filter to consistent + pace-matched → strongest edge

Run after model_v2.py while bt (test set) and the full merged/ats_df
(Vegas comparison) DataFrames are in memory.
"""

import numpy as np
import pandas as pd

# ============================================================
# PART 1: Compute pace mismatch and consistency for each test game
# ============================================================

# bt should have: team, opponent, season, base_spread, full_pred, mov,
#                  opp_str_adj, pace_adj, reversion_adj, consistency_mult

# Reconstruct the underlying consistency values from consistency_mult
# consistency_mult = 1 - BLEND + BLEND * avg_con
# With BLEND = 0.3: avg_con = (consistency_mult - 0.7) / 0.3
# But if we're using the recommended BLEND=0.0, consistency_mult is all 1.0
# In that case we need to recompute. Let's do it from scratch.

print("=" * 70)
print("TESTING PACE & CONSISTENCY AS BET FILTERS")
print("=" * 70)

# We need the team adjustments dict. If not available, we'll compute
# proxies from the bt data itself. But ideally we should have the
# adjustments. Let's work with what we have in bt.

# For this analysis, we'll work with the Vegas comparison (merged/ats_df)
# since that's where ATS matters. If those aren't available, we'll use bt.

try:
    test_df = ats_df.copy()
    has_vegas = True
    print("  Using Vegas comparison data (ats_df)")
except NameError:
    test_df = bt.copy()
    test_df['edge'] = test_df['full_pred'].abs()  # proxy
    test_df['model_ats_correct'] = 1.0  # placeholder
    has_vegas = False
    print("  Using bt (no Vegas data available)")

# ============================================================
# PART 2: Recompute per-team consistency from the rated data
# ============================================================

# We need the team_adjustments dict from the model run.
# If current_adjustments is in memory, great. Otherwise recompute.
# For robustness, let's recompute from bt itself.

# For each team-season, compute residual std from their test appearances
# This is a proxy — ideally we'd use the full prior rated data
# But bt has base_spread and mov, so base_resid is available.

print("\n  Computing team consistency and pace metrics...")

bt['base_resid'] = bt['mov'] - bt['base_spread']

team_stats = {}
for (team, season), grp in bt.groupby(['team', 'season']):
    resids = grp['base_resid'].values
    if len(resids) >= 5:
        team_stats[(team, season)] = {
            'resid_std': np.std(resids),
            'resid_mean': np.mean(resids),
            'n_games': len(resids),
        }

# Normalize resid_std to percentile within each season
for season in bt['season'].unique():
    season_teams = {k: v for k, v in team_stats.items() if k[1] == season}
    if not season_teams:
        continue
    stds = [v['resid_std'] for v in season_teams.values()]
    stds_sorted = sorted(stds)
    for k in season_teams:
        # Percentile rank (0 = most consistent, 1 = most volatile)
        pct = stds_sorted.index(season_teams[k]['resid_std']) / max(len(stds_sorted) - 1, 1)
        team_stats[k]['volatility_pct'] = pct
        team_stats[k]['consistency'] = 1.0 - pct  # higher = more consistent

# ============================================================
# PART 3: Attach to test games
# ============================================================

# For BOTH bt and merged/ats_df, attach team/opp consistency
for df_name, df in [('bt', bt)] + ([('ats_df', ats_df), ('merged', merged)] if has_vegas else []):
    df['team_consistency'] = df.apply(
        lambda r: team_stats.get((r['team'], r['season']), {}).get('consistency', 0.5),
        axis=1
    )
    df['opp_consistency'] = df.apply(
        lambda r: team_stats.get((r['opponent'], r['season']), {}).get('consistency', 0.5),
        axis=1
    )
    df['avg_consistency'] = (df['team_consistency'] + df['opp_consistency']) / 2
    df['min_consistency'] = df[['team_consistency', 'opp_consistency']].min(axis=1)

# Pace mismatch: how different are the two teams' preferred tempos?
# We have this in the rated data. For bt, we can reconstruct from the prediction:
# expected_poss = (team_tempo + opp_tempo) / 2
# pace_dev_a = expected_poss - team_tempo → team_tempo = expected_poss - pace_dev_a
# But we don't have these stored separately. Let's use a proxy:
# The pace_adj magnitude reflects how unusual the pace is for both teams.
# Larger |pace_adj| → bigger mismatch.

# Actually, we stored pace_adj in bt. Its magnitude IS the pace mismatch signal.
# For the Vegas comparison, we need to merge this back in.
if has_vegas:
    # Merge pace info from bt → merged via gameId
    pace_info = bt[['gameId', 'pace_adj']].drop_duplicates('gameId')
    for df_name, df in [('ats_df', ats_df), ('merged', merged)]:
        if 'pace_adj' not in df.columns:
            df_temp = df.merge(pace_info, on='gameId', how='left', suffixes=('', '_bt'))
            df['pace_adj'] = df_temp['pace_adj'].fillna(0)
    
    ats_df['abs_pace_mismatch'] = ats_df['pace_adj'].abs()
    merged['abs_pace_mismatch'] = merged['pace_adj'].abs()

bt['abs_pace_mismatch'] = bt['pace_adj'].abs()


# ============================================================
# PART 4: Test consistency as a bet filter (on bt first)
# ============================================================

print("\n" + "=" * 70)
print("CONSISTENCY AS PREDICTION QUALITY FILTER (on bt)")
print("=" * 70)

# Apply the optimal params from sweep: REV_W=0.48, OPP_SHR=0.80, PACE=off
CURRENT_OPP_KEEP = 1 - 0.85  # what model_v2 used
CURRENT_REV_W = 0.10

optimal_rev_scale = 0.48 / CURRENT_REV_W
optimal_opp_scale = (1 - 0.80) / CURRENT_OPP_KEEP

bt['optimal_pred'] = (bt['base_spread']
                      + bt['reversion_adj'] * optimal_rev_scale
                      + bt['opp_str_adj'] * optimal_opp_scale)
bt['optimal_err'] = (bt['mov'] - bt['optimal_pred']).abs()

print(f"\n  {'Filter':<35s} {'MAE':>7} {'Corr':>7} {'SU%':>6} {'N':>6}")
print(f"  {'-'*70}")

for label, mask in [
    ('All games',                   pd.Series(True, index=bt.index)),
    ('avg_consistency >= 0.3',      bt['avg_consistency'] >= 0.3),
    ('avg_consistency >= 0.4',      bt['avg_consistency'] >= 0.4),
    ('avg_consistency >= 0.5',      bt['avg_consistency'] >= 0.5),
    ('avg_consistency >= 0.6',      bt['avg_consistency'] >= 0.6),
    ('avg_consistency >= 0.7',      bt['avg_consistency'] >= 0.7),
    ('min_consistency >= 0.3',      bt['min_consistency'] >= 0.3),
    ('min_consistency >= 0.4',      bt['min_consistency'] >= 0.4),
    ('min_consistency >= 0.5',      bt['min_consistency'] >= 0.5),
    ('min_consistency >= 0.6',      bt['min_consistency'] >= 0.6),
]:
    sub = bt[mask]
    if len(sub) < 50:
        continue
    mae = sub['optimal_err'].mean()
    corr = sub['optimal_pred'].corr(sub['mov'])
    su = ((sub['optimal_pred'] > 0) == (sub['mov'] > 0)).mean() * 100
    print(f"  {label:<35s} {mae:>7.3f} {corr:>7.4f} {su:>5.1f}% {len(sub):>5}")


# ============================================================
# PART 5: Test pace mismatch as prediction quality filter (on bt)
# ============================================================

print(f"\n  {'Filter':<35s} {'MAE':>7} {'Corr':>7} {'SU%':>6} {'N':>6}")
print(f"  {'-'*70}")

# Pace mismatch percentiles
pace_pcts = bt['abs_pace_mismatch'].quantile([0.25, 0.5, 0.75]).values

for label, mask in [
    ('All games',                          pd.Series(True, index=bt.index)),
    (f'pace_mismatch <= p25 ({pace_pcts[0]:.2f})', bt['abs_pace_mismatch'] <= pace_pcts[0]),
    (f'pace_mismatch <= p50 ({pace_pcts[1]:.2f})', bt['abs_pace_mismatch'] <= pace_pcts[1]),
    (f'pace_mismatch <= p75 ({pace_pcts[2]:.2f})', bt['abs_pace_mismatch'] <= pace_pcts[2]),
    ('pace_mismatch == 0 (no adj)',        bt['abs_pace_mismatch'] < 0.01),
]:
    sub = bt[mask]
    if len(sub) < 50:
        continue
    mae = sub['optimal_err'].mean()
    corr = sub['optimal_pred'].corr(sub['mov'])
    su = ((sub['optimal_pred'] > 0) == (sub['mov'] > 0)).mean() * 100
    print(f"  {label:<35s} {mae:>7.3f} {corr:>7.4f} {su:>5.1f}% {len(sub):>5}")


# ============================================================
# PART 6: COMBINED FILTERS on Vegas ATS data (the money test)
# ============================================================

if has_vegas:
    print("\n" + "=" * 70)
    print("VEGAS ATS WITH CONSISTENCY + PACE FILTERS")
    print("=" * 70)

    # Rebuild optimal predictions for merged/ats_df
    # We need opp_str_adj and reversion_adj on merged. Merge from bt if needed.
    adj_cols = ['gameId', 'opp_str_adj', 'reversion_adj', 'base_spread']
    adj_info = bt[adj_cols].drop_duplicates('gameId')
    
    # For the vegas predictions (wf), we ran a separate expanding_window_predict
    # which already has these columns. Check if ats_df has them.
    if 'opp_str_adj' not in ats_df.columns:
        print("  NOTE: ats_df missing adjustment columns. Using full_pred as-is.")
        ats_df['optimal_pred'] = ats_df['full_pred']
    else:
        ats_df['optimal_pred'] = (ats_df['base_spread']
                                   + ats_df['reversion_adj'] * optimal_rev_scale
                                   + ats_df['opp_str_adj'] * optimal_opp_scale)

    ats_df['optimal_margin'] = ats_df['optimal_pred'] - ats_df['vegas_margin']
    ats_df['optimal_ats'] = (
        np.sign(ats_df['optimal_margin']) == np.sign(ats_df['actual_vs_spread'])
    ).astype(float)
    ats_df['optimal_edge'] = ats_df['optimal_margin'].abs()

    # Filter combos
    print(f"\n  {'Filter':<45s} {'ATS%':>6} {'Record':>12} {'N':>6}")
    print(f"  {'-'*75}")

    filters = [
        ('All games', pd.Series(True, index=ats_df.index)),
        ('avg_consistency >= 0.4', ats_df['avg_consistency'] >= 0.4),
        ('avg_consistency >= 0.5', ats_df['avg_consistency'] >= 0.5),
        ('avg_consistency >= 0.6', ats_df['avg_consistency'] >= 0.6),
        ('min_consistency >= 0.3', ats_df['min_consistency'] >= 0.3),
        ('min_consistency >= 0.4', ats_df['min_consistency'] >= 0.4),
        ('min_consistency >= 0.5', ats_df['min_consistency'] >= 0.5),
    ]

    for label, mask in filters:
        sub = ats_df[mask]
        if len(sub) < 30:
            continue
        wins = int(sub['optimal_ats'].sum())
        losses = len(sub) - wins
        pct = sub['optimal_ats'].mean() * 100
        print(f"  {label:<45s} {pct:>5.1f}% {wins:>5}-{losses:<5} {len(sub):>5}")

    # Now combine consistency with edge thresholds
    print(f"\n  --- Consistency + Edge combined ---")
    print(f"  {'Filter':<45s} {'ATS%':>6} {'Record':>12} {'N':>6}")
    print(f"  {'-'*75}")

    for con_thresh in [0.0, 0.3, 0.4, 0.5]:
        for edge_thresh in [0, 1, 2, 3, 5]:
            if con_thresh == 0.0:
                label = f"edge >= {edge_thresh}"
                mask = ats_df['optimal_edge'] >= edge_thresh
            else:
                label = f"avg_con >= {con_thresh} & edge >= {edge_thresh}"
                mask = ((ats_df['avg_consistency'] >= con_thresh) &
                        (ats_df['optimal_edge'] >= edge_thresh))
            
            sub = ats_df[mask]
            if len(sub) < 30:
                continue
            wins = int(sub['optimal_ats'].sum())
            losses = len(sub) - wins
            pct = sub['optimal_ats'].mean() * 100
            marker = " ✓" if pct >= 52.4 else ""
            print(f"  {label:<45s} {pct:>5.1f}% {wins:>5}-{losses:<5} {len(sub):>5}{marker}")
        print()

    # ============================================================
    # PART 7: Per-season breakdown of best filter combos
    # ============================================================
    print("\n" + "=" * 70)
    print("PER-SEASON: BEST FILTER COMBOS")
    print("=" * 70)

    # Test the most promising combos per season
    combos_to_test = [
        ('No filter', pd.Series(True, index=ats_df.index)),
        ('con>=0.4 & edge>=2', (ats_df['avg_consistency'] >= 0.4) & (ats_df['optimal_edge'] >= 2)),
        ('con>=0.5 & edge>=2', (ats_df['avg_consistency'] >= 0.5) & (ats_df['optimal_edge'] >= 2)),
        ('con>=0.4 & edge>=3', (ats_df['avg_consistency'] >= 0.4) & (ats_df['optimal_edge'] >= 3)),
        ('con>=0.5 & edge>=3', (ats_df['avg_consistency'] >= 0.5) & (ats_df['optimal_edge'] >= 3)),
    ]

    for label, mask in combos_to_test:
        print(f"\n  {label}:")
        print(f"  {'Season':<8} {'ATS%':>6} {'Record':>12} {'N':>6}")
        print(f"  {'-'*40}")
        for season in sorted(ats_df['season'].unique()):
            sub = ats_df[mask & (ats_df['season'] == season)]
            if len(sub) < 10:
                continue
            wins = int(sub['optimal_ats'].sum())
            losses = len(sub) - wins
            pct = sub['optimal_ats'].mean() * 100
            print(f"  {season:<8} {pct:>5.1f}% {wins:>5}-{losses:<5} {len(sub):>5}")
        sub = ats_df[mask]
        if len(sub) >= 30:
            wins = int(sub['optimal_ats'].sum())
            losses = len(sub) - wins
            pct = sub['optimal_ats'].mean() * 100
            print(f"  {'-'*40}")
            print(f"  {'ALL':<8} {pct:>5.1f}% {wins:>5}-{losses:<5} {len(sub):>5}")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print("""
Key findings:
1. Does consistency predict which games we'll be more accurate on?
2. Does pace mismatch predict prediction noise?
3. Can we find a consistency + edge combo that clears 52.4% ATS?
4. Is the edge consistent across seasons? (critical for real betting)
""")

TESTING PACE & CONSISTENCY AS BET FILTERS
  Using Vegas comparison data (ats_df)

  Computing team consistency and pace metrics...

CONSISTENCY AS PREDICTION QUALITY FILTER (on bt)

  Filter                                  MAE    Corr    SU%      N
  ----------------------------------------------------------------------
  All games                             9.741  0.4810  66.0% 11242
  avg_consistency >= 0.3                8.731  0.5477  68.1%  9184
  avg_consistency >= 0.4                8.286  0.5720  68.8%  7560
  avg_consistency >= 0.5                7.886  0.5918  70.3%  5726
  avg_consistency >= 0.6                7.174  0.6348  71.8%  3461
  avg_consistency >= 0.7                6.600  0.6624  74.7%  1826
  min_consistency >= 0.3                8.387  0.5620  68.8%  6277
  min_consistency >= 0.4                8.197  0.5711  69.0%  4850
  min_consistency >= 0.5                7.956  0.5763  69.8%  3703
  min_consistency >= 0.6                6.886  0.6458  75.1%  1453

  Filt